#### Import libraries

In [139]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme(style="whitegrid", palette="muted")

#### Load dataset

In [192]:
train = pd.read_csv("../data/Mobile_Price_Prediction_train.csv")
test = pd.read_csv("../data/Mobile_Price_Prediction_test.csv")

print(f"Train: {train.shape} | Test: {test.shape}")

Train: (2000, 22) | Test: (1000, 24)


In [193]:
train.head()

,id,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,1,707,No,2.1,Yes,2,0,25,0.8,Med,...,495,574,3838,9,2,7,1,No,Yes,3
1,2,1218,No,1.8,Yes,1,1,14,0.5,Med,...,229,616,398,17,0,5,1,No,No,0
2,3,1986,No,0.5,No,7,0,51,0.4,Low,...,342,774,3165,5,0,13,1,No,No,3
3,4,1290,No,0.5,No,0,0,3,0.1,Med,...,600,1109,1214,9,8,15,0,Yes,Yes,1
4,5,1135,Yes,2.8,Yes,9,0,43,0.4,Med,...,690,1589,3204,18,13,6,1,No,No,3


In [194]:
test.head()

,id,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,...,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,Unnamed: 21,Unnamed: 22,Unnamed: 23
0,1,1043,Yes,1.8,Yes,14,0,5,0.1,High,...,3476,12,7,2,0,Yes,No,NaN,NaN,NaN
1,2,841,Yes,0.5,Yes,4,1,61,0.8,High,...,3895,6,0,7,1,No,No,NaN,NaN,NaN
2,3,1807,Yes,2.8,No,1,0,27,0.9,High,...,2396,17,10,10,0,Yes,Yes,NaN,,NaN
3,4,1546,No,0.5,Yes,18,1,25,0.5,Low,...,3893,10,0,7,1,Yes,No,NaN,NaN,NaN
4,5,1434,No,1.4,No,11,1,49,0.5,Low,...,1773,15,8,7,1,No,Yes,NaN,NaN,NaN


#### Feature Description

**Attributes**  
| Type | Columns |
| --------------------- | ---------------------------------------------------------------|
| Numeric |  `id`, `battery_power`, `clock_speed`, `fc`, `four_g`, `int_memory`, `m_dep`, `n_cores`, `pc`, `px_height`, `px_width`, `ram`, `sc_h`, `sc_w`, `talk_time`, `three_g`, `price_range` |
| Categorical (nominal) | `blue`, `dual_sim`, `touch_screen`, `wifi` |
| Categorical (ordinal) | `mobile_wt` {Low < Med < High} |
| **Target** | `price_range` = 0 / 1 / 2 / 3 | 

#### Data Quality Audit  
Check types, missing values, duplicates, impossible values and how the categories are spelled before any modelling.

In [195]:
audit = pd.DataFrame({
    "dtype": train.dtypes,
    "missing": train.isna().sum(),
    "n_unique": train.nunique(),
    "example": train.iloc[0]
})

display(audit)
print(f"Missing values in total: {int(train.isna().sum().sum())}")
print(f"Fully duplicated rows: {int(train.duplicated().sum())}")

,dtype,missing,n_unique,example
id,int64,0,2000,1
battery_power,int64,0,1094,707
blue,str,0,2,No
clock_speed,float64,0,26,2.1
dual_sim,str,0,2,Yes
fc,int64,0,20,2
four_g,int64,0,2,0
int_memory,int64,0,63,25
m_dep,float64,0,10,0.8
mobile_wt,str,0,3,Med


Missing values in total: 0
Fully duplicated rows: 0


In [196]:
# Drop id from train dataset
train = train.drop(["id"], axis=1)

In [197]:
num_cols_all = [c for c in train.select_dtypes("number").columns if c not in ["price_range"]]
cat_cols_all = [c for c in train.columns if c not in num_cols_all if c not in ["price_range"]]

print(f"Numeric columns: {num_cols_all}")
print(f"Categorical columns: {cat_cols_all}\n")

# Convert categorical plain strings to objects {robust for imputers/ encoders}
for c in cat_cols_all:
    train[c] = train[c].astype(object)

# List the values of categorical features
for c in cat_cols_all:
    print(f"{c:15s} {train[c].value_counts().to_dict()}")

print("\n Numeric ranges:")
display(train[num_cols_all].describe().T)

Numeric columns: ['battery_power', 'clock_speed', 'fc', 'four_g', 'int_memory', 'm_dep', 'n_cores', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g']
Categorical columns: ['blue', 'dual_sim', 'mobile_wt', 'touch_screen', 'wifi']

blue            {'No': 1010, 'Yes': 990}
dual_sim        {'Yes': 1019, 'No': 981}
mobile_wt       {'Med': 826, 'Low': 678, 'High': 496}
touch_screen    {'Yes': 1006, 'No': 994}
wifi            {'Yes': 1014, 'No': 986}

 Numeric ranges:


,count,mean,std,min,25%,50%,75%,max
battery_power,2000.0,1238.51850,439.418206,501.0,851.75,1226.0,1615.25,1998.0
clock_speed,2000.0,1.52225,0.816004,0.5,0.70,1.5,2.20,3.0
fc,2000.0,4.30950,4.341444,0.0,1.00,3.0,7.00,19.0
four_g,2000.0,0.52150,0.499662,0.0,0.00,1.0,1.00,1.0
int_memory,2000.0,32.04650,18.145715,2.0,16.00,32.0,48.00,64.0
m_dep,2000.0,0.50175,0.288416,0.1,0.20,0.5,0.80,1.0
n_cores,2000.0,4.52050,2.287837,1.0,3.00,4.0,7.00,8.0
pc,2000.0,9.91650,6.064315,0.0,5.00,10.0,15.00,20.0
px_height,2000.0,645.10800,443.780811,0.0,282.75,564.0,947.25,1960.0
px_width,2000.0,1251.51550,432.199447,500.0,874.75,1247.0,1633.00,1998.0


**What the audit tells us**: The train dataset has no misssing values and no duplicates, so no rows are dropped. Categories are consistent. Numeric ranges are plausible.